# Clase 1 — Introducción a la IA avanzada y tu primera red neuronal

**Curso: Advanced Deep Learning**

## Objetivos de esta clase

- Sitúas el aprendizaje profundo (*deep learning*, DL) avanzado dentro del panorama de la inteligencia artificial (IA) y conoces las arquitecturas principales que vas a estudiar en el curso
- Construyes con NumPy tu primera neurona artificial y una capa completa
- Calculas a mano el gradiente de una red mínima y confirmas el resultado con PyTorch y su motor de diferenciación automática (*autograd*)
- Defines, entrenas y evalúas tu primer perceptrón multicapa (MLP) en PyTorch sobre un problema no lineal
- Sales de esta clase con el código base que reutilizarás durante el resto del curso

**Acrónimos de esta clase:** IA (inteligencia artificial) · DL (*deep learning*) · ANN (red neuronal artificial) · MLP (perceptrón multicapa) · CNN (red neuronal convolucional) · RNN (red neuronal recurrente) · GAN (red generativa antagónica) · VAE (autocodificador variacional) · NLP (procesamiento del lenguaje natural) · SGD (descenso de gradiente estocástico)

## El mapa del curso — ¿qué es el deep learning avanzado?

- El *deep learning* es un subcampo del aprendizaje automático (*machine learning*) que usa redes neuronales con muchas capas para extraer patrones directamente de datos en bruto, sin diseñar a mano las características de entrada
- "Profundo" se refiere al número de capas apiladas, no a ninguna sofisticación conceptual
- Casos reales: AlphaZero aprende ajedrez jugando contra sí mismo; AlphaFold predice la forma tridimensional de proteínas con precisión cercana a los métodos experimentales

### Arquitecturas que verás en el curso

| Arquitectura | Qué resuelve | Idea clave |
|---|---|---|
| Redes convolucionales (CNN) | Datos con estructura espacial (imágenes) | Filtros locales reutilizables |
| Redes recurrentes (RNN) | Datos secuenciales (texto, series temporales) | Estado oculto que recuerda el pasado |
| Atención / Transformer | Dependencias entre elementos de una secuencia | La red "elige" en qué fijarse |
| Redes generativas antagónicas (GAN) | Crear datos nuevos y realistas | Generador contra discriminador |
| Autocodificadores variacionales (VAE) | Aprender variables latentes | Comprimir y reconstruir |
| IA responsable | Robustez, sesgo, impacto social | Parte integral, no un añadido |

- Hoy construyes la pieza común a todas ellas: la neurona artificial y su entrenamiento mediante retropropagación (*backpropagation*)

## Tu primera neurona

- Una neurona artificial recibe entradas, las multiplica por pesos que reflejan su importancia, suma un sesgo (*bias*) y pasa el resultado por una función de activación
- Fórmula: `z = w·x + b` ; `salida = f(z)`
- Sin una función de activación no lineal, apilar capas no aporta nada: el conjunto seguiría siendo una transformación afín
- Usamos la sigmoide, que comprime cualquier valor en el rango (cero, uno) y se interpreta como una probabilidad

In [ ]:
import numpy as np

def sigmoide(z):
    """Comprime cualquier valor real en el rango (0, 1)."""
    return 1 / (1 + np.exp(-z))

def neurona(x, w, b):
    z = np.dot(x, w) + b   # suma ponderada
    return sigmoide(z)

# Ejemplo: ¿aprobamos un préstamo?
# x1 = ingresos (0=bajo, 1=alto), x2 = historial de pagos (0=malo, 1=excelente)
x = np.array([0.8, 0.9])
w = np.array([0.4, 1.2])   # el historial pesa más que los ingresos
b = -0.9

salida = neurona(x, w, b)
print(f"Salida: {salida:.3f} -> {'aprobado' if salida > 0.5 else 'rechazado'}")

# Perfil de riesgo
x_riesgo = np.array([0.2, 0.3])
print(f"Salida (riesgo): {neurona(x_riesgo, w, b):.3f}")

In [ ]:
# Una capa es simplemente varias neuronas en paralelo:
# cada fila de la matriz W son los pesos de una neurona distinta.
def capa(x, W, b):
    z = W @ x + b           # producto matricial + sesgos
    return sigmoide(z)

W_capa = np.array([
    [0.4, 1.2],    # neurona 1 (igual que antes)
    [1.0, -0.5],   # neurona 2 (prioriza ingresos altos)
])
b_capa = np.array([-0.9, -0.1])

print("Salidas de la capa:", capa(x, W_capa, b_capa))

## Backpropagation a mano

- La red aprende ajustando pesos para reducir una función de pérdida (*loss*)
- *Backpropagation* aplica la regla de la cadena para calcular cuánto contribuye cada peso al error final
- Ejemplo mínimo: una neurona con un solo peso `w`, sin sesgo, activación sigmoide y pérdida cuadrática
- `ŷ = σ(w·x)` ; `pérdida = (ŷ − y_objetivo)²`
- El gradiente se descompone en tres derivadas encadenadas: pérdida→ŷ, ŷ→z, z→w

In [ ]:
x = 2.0
y_objetivo = 1.0
w = 0.3       # peso inicial (normalmente sería aleatorio)
eta = 0.1     # tasa de aprendizaje

def derivada_sigmoide(z):
    s = sigmoide(z)
    return s * (1 - s)   # propiedad de la sigmoide

for paso in range(5):
    # --- pase hacia adelante ---
    z = w * x
    y_pred = sigmoide(z)
    perdida = (y_pred - y_objetivo) ** 2

    # --- pase hacia atrás: regla de la cadena, término a término ---
    dL_dypred = 2 * (y_pred - y_objetivo)
    dypred_dz = derivada_sigmoide(z)
    dz_dw = x
    gradiente_w = dL_dypred * dypred_dz * dz_dw

    # --- descenso de gradiente ---
    w = w - eta * gradiente_w

    print(f"paso {paso}: w={w:.5f}  y_pred={y_pred:.5f}  "
          f"perdida={perdida:.5f}  gradiente={gradiente_w:.5f}")

## PyTorch y autograd

- PyTorch calcula los gradientes por ti con su motor de diferenciación automática, *autograd*
- `requires_grad=True` marca un tensor como parámetro entrenable
- `perdida.backward()` recorre el grafo computacional hacia atrás, igual que hiciste a mano
- El optimizador aplica la fórmula del descenso de gradiente por ti
- Compara los números con la celda anterior: deben coincidir paso a paso

In [ ]:
import torch

# Mismos datos que en la versión a mano
x_t = torch.tensor(2.0)
y_objetivo_t = torch.tensor(1.0)

# requires_grad=True: PyTorch registra las operaciones para poder derivarlas después
w_t = torch.tensor(0.3, requires_grad=True)

eta = 0.1
optimizador = torch.optim.SGD([w_t], lr=eta)

for paso in range(5):
    # --- pase hacia adelante ---
    z = w_t * x_t
    y_pred = torch.sigmoid(z)
    perdida = (y_pred - y_objetivo_t) ** 2

    # --- pase hacia atrás: autograd calcula dL/dw automáticamente ---
    optimizador.zero_grad()
    perdida.backward()

    print(f"paso {paso}: w={w_t.item():.5f}  y_pred={y_pred.item():.5f}  "
          f"perdida={perdida.item():.5f}  gradiente={w_t.grad.item():.5f}")

    # --- descenso de gradiente, aplicado por el optimizador ---
    optimizador.step()

## Tu primer perceptrón multicapa

- Un perceptrón multicapa (MLP) apila varias capas de neuronas. La capa oculta transforma el espacio de entrada hasta volverlo separable
- El problema XOR (or exclusivo) es el ejemplo clásico que un modelo lineal no puede resolver, pero sí dos neuronas en una capa oculta
- Construyes el MLP con `torch.nn`, defines una pérdida y un optimizador, y repites el ciclo de siempre: hacia adelante, `backward()`, `step()`
- Al final, visualizas la curva de pérdida para comprobar que la red aprende

In [ ]:
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# Problema XOR: la salida es verdadera solo cuando exactamente una entrada vale 1
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.oculta = nn.Linear(2, 4)   # capa oculta: 2 entradas -> 4 neuronas
        self.salida = nn.Linear(4, 1)   # capa de salida: 4 -> 1

    def forward(self, x):
        h = torch.relu(self.oculta(x))
        return torch.sigmoid(self.salida(h))

modelo = MLP()
perdida_fn = nn.BCELoss()   # entropía cruzada binaria, para clasificación con 2 clases
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.05)

In [ ]:
historial_perdida = []

for epoca in range(500):
    y_pred = modelo(X)
    perdida = perdida_fn(y_pred, Y)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

    historial_perdida.append(perdida.item())

print("Predicciones finales:")
for entrada, objetivo, pred in zip(X, Y, modelo(X)):
    print(f"{entrada.tolist()} -> objetivo={objetivo.item():.0f}  prediccion={pred.item():.3f}")

In [ ]:
plt.plot(historial_perdida)
plt.xlabel("Época")
plt.ylabel("Pérdida (BCE)")
plt.title("Curva de pérdida del MLP sobre XOR")
plt.show()

## Resumen de la clase

- Una neurona artificial es una suma ponderada más una función de activación no lineal
- Apilar capas construye representaciones jerárquicas: de patrones simples a conceptos abstractos
- *Backpropagation* calcula el gradiente de cada peso aplicando la regla de la cadena, hacia atrás en el grafo computacional
- PyTorch automatiza ese cálculo con *autograd*: `requires_grad=True`, `.backward()` y un optimizador
- Un MLP con al menos una capa oculta resuelve problemas no lineales como XOR, algo imposible para una sola neurona
- Este es el mecanismo base que reutilizan las CNN, las RNN, el Transformer, las GAN y los VAE que estudiarás en las próximas clases

**Abre ahora el notebook de práctica y pon en marcha lo que acabas de ver: `../notebooks_practica/clase01_practica_intro_fundamentos_redes_neuronales.ipynb`**